# 03 · Prompt variants

**Question:** does a prompt change improve triage without losing Critical recall?

The first Nova Lite run marked 13 messages P1 where the rubric says P2: a
vulnerable person with water still at knee or waist level.

- `triage-p2-boundary` spells that boundary out. It raised Critical precision
  from 70.2% to 93.9% but dropped Critical recall from 97.1% to 91.2%: the
  boundary overrode a bleeding head injury and a worsening update.
- `triage-p2-boundary-v2` keeps the boundary but lists what is always P1
  (medical emergencies, injured people, worsening updates) before it.

Variants live in `notebooks/prompts/`. Only a winner is copied to
`api/src/saklolo/agents/prompts/`.

In [ ]:
import lab

from saklolo.agents.prompts import PromptSet

In [ ]:
run_settings = lab.settings()
default = PromptSet.default()
variants = {"default": default} | {
    name: PromptSet(extract=default.extract, triage=lab.prompt_variant(name))
    for name in ["triage-p2-boundary", "triage-p2-boundary-v2"]
}

In [ ]:
runs = {}
for label, prompts in variants.items():
    runs[label] = lab.run(lab.build_deps(run_settings, prompts=prompts), run_settings)
    print(label, "done,", len(runs[label].failed), "failed")

## Results
A variant that lowers Critical recall is rejected, even if precision improves.

In [ ]:
lab.compare(runs)

## Messages whose priority changed

In [ ]:
lab.priority_changes(runs["default"], runs["triage-p2-boundary-v2"])

## Decision
Write whether the variant wins and why. To promote it, copy it over
`api/src/saklolo/agents/prompts/triage.md` and run `make eval` (see `notebooks/README.md`).